# Viva Preparation — Supervisor Questions & Answers

**Thesis:** *When Does Kubernetes Become Worth It?*  
**Author:** Sirajulhaq Wahaj | **Supervisor:** Ludvig Malm | **Date:** April 2026

---

**How to use this notebook**

Each question is one cell. The answer follows immediately below it. Before reading the answer, try to answer the question yourself in your head. If you hesitate or miss something, mark that question with ⚠️ and return to it.

Questions are ordered from **basic → hard** within each section. Questions marked 🔴 are the highest-risk ones — they probe genuine weaknesses in the study.

| Symbol | Meaning |
|--------|--------|
| 🟢 | Straightforward — you must answer this fluently |
| 🟡 | Medium — requires connecting two ideas |
| 🔴 | Hard — probes a real limitation or ambiguity |

---
## Part 1 — Research Question and Motivation

### 🟢 Q1. State the main research question in one sentence. Why does it matter to a practising DevOps engineer?

**Answer:**

*At what concurrency level does migrating a Dagster pipeline orchestration system from a VM DockerRunLauncher to a Kubernetes K8sRunLauncher become net beneficial in terms of reliability and execution performance?*

It matters because every team running workflow orchestration on a single VM eventually asks this exact question. Before this study, there was no empirical, application-level answer — only infrastructure benchmarks measuring container overhead in isolation, disconnected from actual job failure rates.

### 🟢 Q2. What is the "crossover point"? Define both the reliability crossover and the performance crossover precisely.

**Answer:**

**Reliability crossover** — the first concurrency level at which the VM's job success rate drops below 95% while Kubernetes maintains 100%. Found at **L3 (3 concurrent jobs)**.

**Performance crossover** — the first concurrency level at which Kubernetes' total wall-clock time per job (execution time + startup overhead) falls below the VM's execution time. Found at **L2 (2 concurrent jobs)**.

Kubernetes is faster than the VM at **L2–L5** in a raw survivor-time comparison, but this becomes misleading at L3+ because the VM is already failing >33% of jobs.

**Combined crossover** — the level where both conditions are simultaneously met. **L3**, where:
- VM success rate = 66.7% (below 95% threshold)
- K8s total time ≈ 69.7 s vs VM survivor time ≈ 69.9 s (effectively equal)
- K8s adds 4–15 s startup overhead but delivers 100% reliability

The reliability crossover is the binding constraint; the performance crossover is secondary.

### 🟡 Q3. Why Dagster? Would results generalise to Airflow or Prefect?

**Answer:**

Dagster was chosen because it natively supports both DockerRunLauncher and K8sRunLauncher through a single configuration change — the same job code, same image, different executor. This makes it uniquely suited to isolate the effect of the execution model from the application code.

**What generalises:**
- The failure *mechanism* (shared-kernel Linux OOM kills) is framework-agnostic. Any containerised Python workflow running on a shared-memory VM will exhibit this behaviour.
- The *shape* of degradation (sharp cliff, not gradual slope) is predicted by resource contention theory and will recur with any framework.

**What does not generalise:**
- The *startup overhead* values (4–15 s). Dagster has a heavier initialisation footprint than, say, Argo Workflows. Lighter runtimes would have lower overhead, shifting the performance crossover earlier.
- The *exact crossover threshold* (L3). This is hardware- and workload-specific.

### 🟡 Q4. What is the precise literature gap? Isn't "VM vs Kubernetes" already well-covered?

**Answer:**

Three independent bodies of literature existed before this thesis:
1. **Resource contention theory** (Nanda 1991, Arora 2023) — predicts non-linear collapse in shared-memory execution
2. **Kubernetes overhead studies** (Casalicchio 2019, Choi 2021) — measures pod scheduling latency and container startup at the infrastructure level
3. **Fault isolation principles** (Čilić 2023) — describes blast radius architecturally

No study connected all three **at the workflow job execution layer** using real pipeline orchestration. Prior work used HTTP benchmarks, CPU stress tests, or generic microservices — never a real orchestration framework (Dagster, Airflow, Prefect) measured by job success rate across concurrency levels.

The specific gap: *no empirical study identified the concurrency threshold at which migration becomes net beneficial for pipeline orchestration*. That decision boundary is the novel contribution.

---
## Part 2 — Experimental Design and Methodology

### 🟢 Q5. Describe the workload. Why SHA-256 hashing + 400 MB NumPy allocation?

**Answer:**

The `thesis_workload` job has two sequential ops:
1. **`cpu_burn`** — runs `hashlib.sha256(b"dagster-thesis-workload").hexdigest()` in a tight loop for exactly 30 seconds. One CPU core at near-100% utilisation. No I/O variability.
2. **`memory_pressure`** — allocates a 400 MB random byte buffer via `np.random.bytes(WORKLOAD_MEMORY_MB * 1024 * 1024)`, keeps that buffer resident, and for the next 30 seconds repeatedly hashes rotating 32-byte slices from it before releasing the memory.

Total: ~60 s per job under no contention.

**Why this design:**
- SHA-256 is deterministic for a given input, CPU-bound, single-core in this usage, and produces reproducible work without introducing I/O variability
- The 400 MB retained byte buffer creates a hard, predictable per-container memory footprint, making contention and OOM behaviour analytically tractable
- Hashing rotating 32-byte slices from that buffer ensures the second phase still performs CPU work while the memory allocation remains live, so Phase 2 represents combined memory pressure plus continued hashing rather than idle allocation
- The two-phase design separates pure CPU contention effects (Phase 1) from memory-associated contention effects (Phase 2); `time.sleep()` would exercise no resources, while matrix multiplication would introduce BLAS/library variability across environments

### 🟢 Q6. What are the six concurrency levels and why were they chosen?

**Answer:**

| Level | Concurrent Jobs | VM Aggregate Memory | Significance |
|-------|----------------|--------------------|--------------|
| L1 | 1 | ~0.4 GB | Pure baseline, no contention |
| L2 | 2 | ~0.8 GB | Minimal contention |
| L3 | 3 | ~1.2 GB | 75% of 4 vCPUs — near CPU saturation |
| L5 | 5 | ~2.0 GB | Over-subscription begins |
| L7 | 7 | ~2.8 GB | Heavy over-subscription |
| L10 | 10 | ~4.0 GB | 100% of VM RAM; 2.5× available CPU |

The non-contiguous numbering (1, 2, 3, 5, 7, 10) concentrates measurements at **inflection points** (L3 = near-CPU-saturation, L5 = over-subscription threshold, L10 = full memory exhaustion) rather than distributing uniformly. This is more informative than L1–L6 sequential levels for detecting failure cliffs.

Each level runs **3 repetitions** with a **60-second cooldown** between repetitions to allow container teardown and PostgreSQL connection pool reset.

### 🔴 Q7. The VM had 4 GB RAM and Kind had 8 GB. This is an unfair comparison — why should we trust your results?

**Answer:**

This is a genuine limitation, explicitly acknowledged in Section 5.6 ("Memory asymmetry between environments"). Here is the honest assessment:

**Why the asymmetry exists:** Multipass on Windows allocates from the host machine's RAM. Both the VM and the Kind cluster must run on the same physical host simultaneously. Giving the VM 8 GB would have left insufficient memory for the Kind cluster at high concurrency levels, making L7–L10 K8s experiments impossible.

**What this means for the results:** The observed crossover at L3 is partly a consequence of the VM having 4 GB. An 8 GB VM would push the OOM threshold to approximately L5–L7. The *specific threshold value* (L3) is therefore hardware-configuration-specific.

**Why the conclusions remain valid:**
1. The *failure mechanism* — shared-kernel OOM kills — operates identically at 4 GB, 8 GB, or 16 GB. Only the level at which it triggers changes.
2. The *architectural lesson* — that per-pod cgroup limits prevent cross-job memory interference — is independent of total capacity.
3. The thesis is explicit that engineers should run this same analysis on their own hardware to find *their* threshold.

The asymmetry weakens the claim that L3 is a universal threshold, but it does not weaken the claim that the failure *mechanism* and *methodology* are valid.

### 🔴 Q8. Three repetitions per level — is this statistically sufficient? Could the results be noise?

**Answer:**

**No, n=3 is not sufficient for formal statistical significance testing.** The thesis acknowledges this in Sections 3.9 and 5.6 and frames all results as *directional findings*.

**Why it is still adequate for the primary finding:**
The success rate at L3 drops from 100% to 66.7%. Over 3 repetitions × 3 jobs each = 9 launched runs, 6 succeeded and 3 failed. The effect size is huge — a 33 percentage point drop. You cannot explain this as noise with n=3.

The binomial probability of observing 6/9 successes (success rate ≥ 66.7%) under a true 100% success rate is essentially zero. The OOM kill mechanism is physically determined (aggregate memory exceeds RAM), not randomly distributed.

**Where n=3 is a real weakness:**
- Characterising startup overhead variance (4–15 s range) — larger n needed for confidence intervals
- Detecting small effects at L1–L2 where both environments are reliable
- Ruling out confounders in execution time measurements

**What would be better:** 6–10 repetitions per level, enabling Mann-Whitney U tests between environments per concurrency level. This is the highest-priority future work item.

### 🟡 Q9. How did you ensure concurrent runs were actually simultaneous and not staggered?

**Answer:**

The test harness (`scripts/trigger_dagster_runs.py`) submits runs in a simple sequential `for` loop, with a small `time.sleep(0.05)` stagger between submissions. So the runs are not launched via parallel threads; instead, they are queued very quickly one after another.

The Dagster GraphQL mutation `launchRun` returns immediately after queuing the run — it does not wait for execution to begin. That means the harness can submit N runs with an approximate spread of `(N-1) × 0.05 s` plus minor request overhead. At the maximum tested level of 10 runs, the intentional stagger is about 0.45 s, so the overall submission window is still typically well under 1 second.

This was sufficient for the experiment because the goal was to create overlapping concurrent workload at the orchestrator and executor level, not to guarantee mathematically identical submission timestamps.

The 60-second cooldown between repetitions serves three purposes:
1. Docker container teardown on the VM (containers exit but their cgroups linger briefly)
2. PostgreSQL connection pool release and reset
3. OS memory reclaim (Linux page cache flush)

Without this cooldown, residual memory pressure from the previous repetition could distort the next run's failure pattern.

### 🔴 Q10. Explain the `job_start_ts` measurement issue in `pod_timing.csv`. What went wrong and how did you fix it?

**Answer:**

**What went wrong:**
`job_start_ts` was intended to capture the moment the Dagster job started executing inside the pod. It was collected from the Kubernetes `ContainersReady` condition's `lastTransitionTime`.

However, `ContainersReady` transitions to `False` (and therefore `lastTransitionTime` updates) when the container **exits** — not when it enters the Ready state. So `job_start_ts` actually recorded the container *exit* timestamp, not the start timestamp.

Using it directly as a startup metric would yield: `job_start_ts − submitted_ts ≈ 67 s` (total pod lifetime), not the startup duration of 4–15 s.

**How it was fixed during analysis:**

The correct startup overhead was computed as:
```
startup_overhead = (job_start_ts − submitted_ts) − (end_time − start_time)
                 = total pod lifetime − Dagster execution time
```

Where `end_time − start_time` comes from `dagster_runs.csv`, joined on `run_id`. This subtraction isolates the time before and after execution — dominated by the startup phase since teardown is near-instantaneous.

**Lesson:** Define all metric calculations before data collection, with explicit field semantics and an example calculation verified against a single known run.

---
## Part 3 — Results

### 🟢 Q11. Summarise the VM results in three sentences.

**Answer:**

The VM maintained 100% job success rate at L1 and L2, then collapsed sharply to 66.7% at L3 — driven by Linux OOM kills when three containers simultaneously allocated 400 MB of memory, exhausting the 4 GB VM's available headroom. Success continued declining to 46.7% at L5, 42.9% at L7, and 30.0% at L10 as more concurrent containers competed for the same fixed memory pool.

Critically, the execution time of *surviving* jobs remained flat across all levels (65–80 s), because OOM kills act as a filter: only the containers that acquired memory before the OOM killer fired completed successfully, and these ran under near-normal conditions.

### 🟢 Q12. Summarise the Kubernetes results in three sentences.

**Answer:**

Kubernetes maintained 100% job success rate across all six concurrency levels (L1–L10), with no failures at any tested level. Execution time grew modestly from 63.4 s at L1 to 81.6 s at L10 — a 29% increase driven by CPU time-sharing on the single-node cluster as concurrency rose — but remained well within acceptable bounds and far below the VM's survivor times at high concurrency.

Container startup time specifically (the Pod Running → job executing interval, not the broader startup-overhead measure reported elsewhere in the thesis) ranged from 4.3 s per job at L1 to 14.9 s at L10, growing with concurrency as more pods competed for PostgreSQL connections and Python initialisation on the single-node Kind cluster.

### 🔴 Q13. VM CPU utilisation *decreased* at higher concurrency levels (60.8% at L2, down to 38% at L10). Explain this.

**Answer:**

This is counter-intuitive but explicable by the **OOM filter effect**.

At L10, 7 of 10 containers were OOM-killed at approximately 37–50 seconds into execution — during the `memory_pressure` op. These containers completed the 30-second `cpu_burn` phase (contributing CPU load) but were killed before the 60-second mark.

The system-wide CPU metric is averaged over the **full wall-clock window** of the experiment (roughly 80–100 seconds). At L10:
- Seconds 0–30: 10 containers burning CPU (~100% of 4 cores)
- Seconds 30–50: 10 containers in memory pressure phase (lower per-container CPU)
- Seconds 50+: 7 containers killed; only 3 remaining

The average CPU over the full window is pulled down by the long tail where only 3 containers are active. It is a measurement artefact of time-averaging, not a real reduction in per-job CPU demand.

A per-job CPU measurement (CPU during active execution only) would show near-constant single-core usage regardless of concurrency level.

### 🔴 Q14. Table 4.5 shows K8s is *slower* than the VM at L7 (+14.5 s) and L10 (+16.6 s). Doesn't this disprove your conclusion that K8s is better?

**Answer:**

No — and this is one of the most important nuances in the thesis.

The +14.5 s delta at L7 compares:
- **K8s total time:** all 21 jobs (7 concurrent jobs × 3 reps), all completing successfully (100% SR) at ~92 s each
- **VM time:** the **surviving** jobs only — the 9 of 21 that were not OOM-killed — at ~77.7 s each

At L7, the VM success rate is **42.9%**. That means 12 of 21 launched jobs produced **zero output** — they were OOM-killed and must be retried. If you account for the failed jobs needing a retry cycle, the effective VM cost per successful job is much higher than 77.7 s.

The comparison is only honest when both numerators are the same population. You cannot compare "K8s time for all successful jobs" against "VM time for the lucky survivors." The right question is: *which system reliably delivers 21 successful jobs?* Only Kubernetes does, at L7.

The thesis correctly uses the **reliability crossover** (not the raw time delta) as the primary decision boundary.

### 🟡 Q15. How do you confirm the VM failures were OOM kills and not Dagster timeouts, DB errors, or network issues?

**Answer:**

Three independent evidence sources:

1. **Failure timing is diagnostic.** OOM-killed jobs terminated at 37–50 s — exactly during the `memory_pressure` op's 400 MB allocation call (~30 s mark). A Dagster run timeout would cut at a configurable cutoff (typically 60+ s). A DB issue would fail at run submission or at the first DB poll, not mid-execution at second 37.

2. **VM memory metrics confirm saturation.** `vm_metrics.csv` records system-wide memory every 1 second. At L3+, memory utilisation reached 80–82% of 4 GB (≈3.2 GB) at the moment of failures — consistent with the Linux kernel invoking the OOM killer at its high-water mark.

3. **Exit code 137.** Docker containers killed by the OOM killer receive `SIGKILL` from the kernel. The OS assigns exit code 137 (= 128 + 9, where 9 is SIGKILL). Container daemon logs on the VM showed exit code 137 for failed runs, not 1 (application error) or 124 (timeout).

All three converge on OOM kills as the sole failure mechanism.

### 🟡 Q16. The blast radius test at L5 showed zero impact in *both* environments. How does this support your SQ2 argument?

**Answer:**

The deliberate blast radius test (kill one pod at L5, observe the other four) is a **lower-bound test** — it shows isolation under controlled conditions. Both environments passed because at L5 the VM still had sufficient memory headroom for the surviving four containers to complete.

The *real* SQ2 evidence is the **organic blast radius at L3–L10** from natural OOM kills:
- On the VM, the Linux OOM killer operates at the system level, not the container level. When aggregate memory exceeds the budget, it can kill multiple containers in the same OOM event — as reflected in the L5 success rate of 46.7% (7 of 15 succeed = ~7 killed across 3 repetitions, often in groups).
- On Kubernetes, cgroup enforcement kills the specific pod exceeding its 2 GiB limit without touching other pods. No cross-pod cascade was observed in the Exp2A data at any level.

The controlled test confirms the isolation mechanism. The organic success rate data (100% K8s vs 30–67% VM at L3–L10) shows the practical consequence at scale.

---
## Part 4 — Discussion and Conclusions

### 🟢 Q17. State the answer to the main research question in plain language — no jargon.

**Answer:**

On a 4-core, 4 GB VM running a memory-intensive workload where each job needs ~1 GB, Kubernetes becomes worth migrating to at **3 concurrent jobs**.

Below 3 concurrent jobs, both systems are reliable and the VM is slightly faster (no Kubernetes startup overhead). At 3 concurrent jobs, the VM starts randomly killing about one in three jobs due to memory shortage. Kubernetes prevents this completely by giving each job a guaranteed, isolated memory allocation.

The startup overhead Kubernetes adds (4–15 seconds per job) is small compared to the reliability it provides. Teams expecting to regularly run 3 or more pipeline jobs at the same time should migrate to Kubernetes. Teams running 1–2 jobs at a time can safely stay on a VM if their memory budget is adequate.

### 🔴 Q18. How confident are you that the L3 crossover generalises beyond your specific hardware? What variables would shift it?

**Answer:**

The specific value **L3 does not generalise** — it is hardware- and workload-specific. The thesis is explicit about this.

**Variables that shift the reliability crossover:**

| Variable | Direction of change | Effect on crossover |
|----------|--------------------|-----------------------|
| VM RAM increases to 8 GB | More headroom | Crossover shifts to ~L5–L7 |
| Per-job memory increases | More pressure per job | Crossover shifts earlier |
| Infrastructure stack memory decreases | More headroom | Crossover shifts later |
| Workload is I/O-bound (not memory-bound) | Different failure mode | OOM may not be primary mechanism |

**What does generalise:**
1. The failure *mechanism* (shared-kernel OOM kills) applies to any Linux-based Docker deployment
2. The *shape* of degradation (discrete cliff, not gradual slope) is universal for memory-driven OOM
3. The *method* — run controlled experiments at increasing concurrency levels on your actual hardware — is directly replicable

The practical contribution is the method, not the specific number. Teams should run their own crossover analysis with their actual workload and hardware.

### 🟡 Q19. Casalicchio (2019) showed containers interfere even with resource limits. You found no interference in Kubernetes. Contradiction?

**Answer:**

Not a contradiction — a scope difference.

Casalicchio's interference result was in the context of CPU-intensive workloads on a **multi-tenant containerised host with HPA autoscaling**. His finding was that response times were 2–3 orders of magnitude higher under the *default HPA* compared to a contention-aware autoscaler. This is an autoscaling result, not a static isolation result.

In this thesis:
1. **No autoscaling** was used — fixed pod counts per level, no HPA
2. **Workload type** — CPU-bound SHA-256 + memory pressure. Casalicchio's interference is most pronounced for I/O-bound and network-bound workloads competing for shared I/O bandwidth, which cgroup limits do not protect
3. **Evidence** — K8s execution time variance was *lower* than VM variance at every concurrency level, directly opposite to what interference would predict

The reconciliation: for **compute-bound workloads with no shared I/O bottleneck**, cgroup CPU and memory limits effectively isolate pods. For **I/O-bound workloads**, Casalicchio's interference mechanism remains relevant even in Kubernetes. Testing with an I/O-bound workload is the most important unresolved question from this study.

### 🟡 Q20. Your hypothesis predicted L5–L7 for the reliability collapse. The actual result was L3. Is this a failure of the hypothesis?

**Answer:**

It is a failure of one assumption in the hypothesis, not the overall framing.

The hypothesis was based on **CPU saturation** as the primary failure mode: 4 vCPUs × 75% (3 jobs) = near saturation. It expected memory pressure to become critical only at L5–L7.

What the hypothesis missed: the **infrastructure stack's resident memory**. Docker daemon + PostgreSQL + Dagster daemon consumed ~1.5–2 GB of the 4 GB VM at rest, before any workload ran. This left only ~2–2.5 GB for containers. Three containers × (400 MB workload + ~500–600 MB Python/Dagster overhead) = ~2.7–3.0 GB — exceeding available headroom immediately at L3.

The hypothesis correctly predicted the *direction* (VM degrades before K8s) and the *mechanism* (shared resource exhaustion). What it got wrong was which resource would be exhausted first (memory, not CPU) and at what level.

This is actually a stronger finding than the hypothesis: **memory exhaustion, not CPU saturation, is the primary reliability risk on a constrained VM**, and it strikes earlier than CPU-focused analysis predicts.

### 🔴 Q21. Two infrastructure fixes were applied mid-experiment. How do you know the reported data is from the stable configuration and not contaminated by the broken one?

**Answer:**

This is acknowledged as a limitation in Section 5.6.

**What was fixed:**
1. Added `dagster-docker` package to the VM Dagster daemon environment (missing → runs could not launch containers)
2. Increased Docker container memory limit from the default to 2 GB (too-low limit → containers OOM-killed by Docker before the job even started, not by the kernel)

**Why the reported data is from the stable configuration:**
- Fix 1 was essential for any runs to complete at all — the broken state produced immediate launch failures, not the gradual OOM pattern seen in the data
- Fix 2 changed where OOM enforcement happened (Docker → Linux kernel). The reported failure pattern (jobs failing at 37–50 s, exit code 137) is consistent with kernel-level OOM kills, not Docker-level enforcement
- The `validate-experiment-setup.sh` pre-flight script was run before each experimental session to confirm both environments matched the expected state

**What cannot be ruled out:** undetected configuration drift from the development phase. This is the honest answer. What would prevent this in future: full Ansible-managed VM state from day one, with every change as a version-controlled playbook commit.

---
## Part 5 — Technical Deep Dive

### 🔴 Q22. Walk through the startup overhead calculation step by step, including the join logic.

**Answer:**

Two data sources are joined on `run_id`:

**From `pod_timing.csv`:**
- `submitted_ts` — timestamp when Dagster submitted the pod to the Kubernetes API
- `job_start_ts` — `ContainersReady lastTransitionTime` (fires on container *exit*, not start — see Q10)
- `(job_start_ts − submitted_ts)` = **total pod lifetime** (creation to completion)

**From `dagster_runs.csv`:**
- `start_time`, `end_time` — Dagster's own timestamps for job execution
- `(end_time − start_time)` = **Dagster execution time** (the time job code was running)

**Join:** `pod_timing.csv JOIN dagster_runs.csv ON run_id`

**Calculation:**
```
startup_overhead = total_pod_lifetime − dagster_execution_time
                 = (job_start_ts − submitted_ts) − (end_time − start_time)
```

This isolates the time spent on pod scheduling + Python interpreter startup + Dagster framework import + PostgreSQL connection establishment — everything that happened *before* the job code began executing. Container teardown is included but is near-instantaneous.

**Example at L1:** total pod lifetime = 67.7 s, Dagster execution = 63.4 s → startup overhead = 4.3 s

### 🟡 Q23. Why does startup overhead grow from 4.3 s at L1 to 14.9 s at L10? What specific bottlenecks cause this?

**Answer:**

Three bottlenecks, all caused by concurrent initialisation on a single-node cluster:

1. **Python package import I/O contention** — Python 3.13 with Dagster, dagster-k8s, numpy, hashlib reads `.pyc` files from the container filesystem layer on startup. When 7–10 containers start simultaneously, they queue for the same node's filesystem I/O bandwidth.

2. **PostgreSQL connection pool queuing** — each run pod opens 1–2 connections to the in-cluster PostgreSQL instance. At L10, 10 pods + Dagster daemon + webserver request ~15–20 connections simultaneously. PostgreSQL processes them serially, creating a queue.

3. **Dagster gRPC readiness handshake** — the Dagster daemon performs a gRPC health check against each run pod's code server before releasing the run. Multiple simultaneous checks at high concurrency queue against the daemon's single-threaded gRPC dispatcher.

Growth is sub-linear (not 10× at L10) because these are partially parallelisable — they don't form a strict serial queue. In a production multi-node cluster, pods start on different nodes, distributing these bottlenecks and reducing overhead growth with concurrency.

### 🟡 Q24. What did `system-reserved` and `kube-reserved` in `kind-config.yaml` actually do? Would removing them change your results?

**Answer:**

The `kind-config.yaml` reserved:
- `system-reserved: cpu=500m, memory=512Mi` — for OS processes outside Kubernetes
- `kube-reserved: cpu=500m, memory=512Mi` — for kubelet and container runtime

Combined: 1 vCPU + 1 GiB RAM removed from the schedulable pool, leaving **~3 vCPU + ~7 GiB** allocatable for workload pods.

**Effect on results:** They prevented the kubelet itself from being OOM-killed at high concurrency. Without these reservations, at L10 (10 pods × ~1 GB actual RSS = ~10 GB total) the node's memory pressure could cause the kubelet to be killed by the OS OOM killer — an uncontrolled failure that would make the experiment unreproducible.

**Would removing them change results?** At L1–L5 — no visible difference. At L7–L10 — possibly yes: node instability could introduce additional random failures and inflate variance. The reservations are a reproducibility safeguard, not a result-shaping parameter.

---
## Part 6 — Limitations and Future Work

### 🟡 Q25. Name the three biggest limitations of this study. For each: what is the impact on the conclusions, and what would fix it?

**Answer:**

**1. Memory asymmetry (VM 4 GB vs Kind 8 GB)**
- *Impact:* The observed crossover at L3 is partly a function of VM memory constraints, not solely execution model architecture. An 8 GB VM would likely push the crossover to L5–L7.
- *Fix:* Provision both environments with equal memory. Run a second experiment set with VM at 8 GB.

**2. Small sample size (n=3 repetitions per level)**
- *Impact:* Startup overhead variance cannot be characterised with confidence intervals. Small effects at L1–L2 cannot be statistically confirmed.
- *Fix:* 6–10 repetitions per level. Enables Mann-Whitney U tests and 95% CIs on all metrics.

**3. Single-node Kind cluster**
- *Impact:* Startup overhead values (4–15 s) are optimistic — production clusters with cold image pulls and multi-node scheduling add 10–60 s. The performance crossover conclusion is a lower bound.
- *Fix:* Repeat K8s experiments on a 3-node GKE cluster with cold image pulls to get production-representative overhead values.

### 🟡 Q26. If you had 6 more months, what is the single most important experiment you would add?

**Answer:**

**Repeat Experiment 1 with an 8 GB VM** — identical to the current Experiment 1 in every respect except VM RAM.

This would:
1. Resolve the memory asymmetry limitation directly
2. Push the OOM threshold higher, likely into the L5–L7 range
3. Reveal whether a pure **CPU-contention crossover** exists (what happens when memory is not the binding constraint)
4. Produce a two-point crossover dataset: one for 4 GB VMs (L3), one for 8 GB VMs (L5–L7 predicted), enabling a generalisation formula: `crossover_level ≈ (available_headroom_GB) / (per_job_memory_GB)`

Second priority: add an I/O-bound workload (100 MB PostgreSQL reads per op) to test whether the Casalicchio interference effect appears in K8s for non-compute workloads.

---
## Part 7 — Reflection

### 🟢 Q27. What is the single most important practical takeaway from this thesis for a DevOps engineer?

**Answer:**

**VM failure under memory-intensive concurrent workloads is not gradual — it is a cliff.** The success rate drops from 100% to 66.7% in one step between L2 and L3. There is no warning signal in execution time metrics (surviving jobs look normal). If you are monitoring only latency, you will not see the failure coming.

**Concrete action:** Add job success rate to your Dagster observability dashboard. If you see the rate drop below 95%, you have crossed your infrastructure's memory threshold and need to either add RAM, reduce concurrent jobs, or migrate to Kubernetes. Do not use execution time alone as the reliability signal.

### 🟢 Q28. What would you do differently if you started this thesis over today?

**Answer:**

Four things, in order of importance:

1. **Match VM and K8s memory (both 8 GB)** — eliminates the hardware asymmetry criticism entirely

2. **Define all metric calculations before collecting data** — the `job_start_ts` measurement issue was discovered during analysis. A one-page schema document with column semantics and a worked example calculation per column would catch this in the design phase

3. **6 repetitions per level from the start** — minimal additional wall time (~2×), but enables proper statistical tests and confidence intervals

4. **Full Ansible-managed VM from day one** — the two mid-experiment infrastructure fixes would appear as version-controlled playbook commits, not undocumented SSH changes

### 🟡 Q29. How does the scientific contribution compare to a university master's thesis?

**Answer:**

Honest comparison:

**Weaker than a university MSc in:** formal statistical inference (no significance tests, small n), theoretical model derivation, generalisability across hardware and workload types

**Comparable to or stronger than a university MSc in:** experimental execution on real infrastructure, connection of multiple bodies of theory to an empirical application-level result, direct practical value (the decision framework answers a question practising engineers actually face)

**Relevant to YH criteria:** A YH thesis is assessed on relevance to professional practice, methodological soundness, and analytical depth — not on theoretical novelty. Against those criteria: high relevance (direct answer to a practitioner question), moderate soundness (justified but limited by hardware and sample size), strong depth (OOM kill mechanism traced from kernel-level through application-level metrics).

The empirical connection between resource contention theory and Dagster job failure behaviour is genuinely novel — no prior study made this specific connection.

### 🔴 Q30. Final question: what is the one question about this thesis that you hope the examiner does NOT ask?

**Answer:**

"Can you demonstrate that the K8s and VM experiments were run under identical host load conditions, given they were on the same physical machine?"

The honest answer: I cannot fully demonstrate this. The experiments were run in separate sessions, but I did not record host-level metrics (CPU, RAM, swap) during both sessions to verify that the physical host was in an identical state. Background processes (Windows updates, browser, IDE) could have consumed different amounts of RAM between the VM session and the K8s session, slightly changing the effective resource available to each experiment.

**What I can say:** The primary finding (VM OOM kills at L3) is physically determined by the workload's memory footprint relative to VM RAM, not by stochastic host variation. The magnitude of the effect (100% → 66.7% success rate drop) is too large to be explained by host background load variation of ±500 MB.

**What I would do differently:** log `Get-Process | Sort-Object -Property WorkingSet -Descending` on the host at the start of each experimental session and commit it alongside the raw data.

---

## Quick-Reference Summary

| # | Question | Level | Section |
|---|----------|-------|---------|
| 1 | Core research question | 🟢 | Intro |
| 2 | Crossover point — both definitions | 🟢 | Intro |
| 3 | Why Dagster / generalisation | 🟡 | Intro |
| 4 | Literature gap — what is novel | 🟡 | Lit Review |
| 5 | Workload design — SHA-256 + 400 MB | 🟢 | Method |
| 6 | Concurrency levels — why non-contiguous | 🟢 | Method |
| 7 | Memory asymmetry 4 GB vs 8 GB | 🔴 | Method |
| 8 | n=3 — statistically sufficient? | 🔴 | Method |
| 9 | Simultaneous submissions — how ensured | 🟡 | Method |
| 10 | `job_start_ts` measurement issue | 🔴 | Method |
| 11 | VM results summary | 🟢 | Results |
| 12 | K8s results summary | 🟢 | Results |
| 13 | CPU utilisation decreasing at L10 | 🔴 | Results |
| 14 | K8s slower at L7/L10 — contradiction? | 🔴 | Results |
| 15 | OOM kill confirmation evidence | 🟡 | Results |
| 16 | Blast radius test interpretation | 🟡 | Results |
| 17 | Answer to RQ in plain language | 🟢 | Conclusions |
| 18 | Generalisability of L3 threshold | 🔴 | Discussion |
| 19 | Casalicchio interference reconciliation | 🟡 | Discussion |
| 20 | Hypothesis revision — L5–L7 → L3 | 🟡 | Discussion |
| 21 | Mid-experiment fixes — data validity | 🔴 | Discussion |
| 22 | Startup overhead calculation walkthrough | 🔴 | Technical |
| 23 | Overhead growth mechanism | 🟡 | Technical |
| 24 | `system-reserved` / `kube-reserved` effect | 🟡 | Technical |
| 25 | Three biggest limitations | 🟡 | Limitations |
| 26 | Most important future experiment | 🟡 | Limitations |
| 27 | Single practical takeaway | 🟢 | Reflection |
| 28 | What would you do differently | 🟢 | Reflection |
| 29 | YH vs university MSc contribution | 🟡 | Reflection |
| 30 | Question you dread most | 🔴 | Reflection |

> **Preparation strategy:** Master 🔴 questions first. These are where examiners find gaps. The 🟢 questions you must answer fluently without hesitation — stumbling on basics undermines confidence in the rest.